In [1]:
import sys, os, subprocess

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)

try:
    import pyspark
    print("PySpark already installed:", pyspark.__version__)
except ImportError:
    print("PySpark NOT installed — install with: !pip install -q pyspark==4.0.4  (then restart kernel)")

print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

Python: 3.13.15
Executable: /usr/bin/python3
PySpark already installed: 4.0.4
openjdk version "21.0.12" 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)



**Step 2: Set environment variables**



In [2]:
import os, sys

os.environ["JAVA_HOME"]           = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_PYTHON"]      = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("JAVA_HOME         :", os.environ["JAVA_HOME"])
print("PYSPARK_PYTHON    :", os.environ["PYSPARK_PYTHON"])

JAVA_HOME         : /usr/lib/jvm/java-21-openjdk-amd64
PYSPARK_PYTHON    : /usr/bin/python3


**Step 3: Create the SparkSession**

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("cipherstop-ingestion")
    .config("spark.driver.memory", "20g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.files.openCostInBytes", str(4 * 1024 * 1024))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark (Python API):", spark.version)
print("Spark (JVM)       :", spark.sparkContext._jsc.sc().version())
print("Master            :", spark.sparkContext.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/21 19:27:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark (Python API): 4.0.4
Spark (JVM)       : 4.0.4
Master            : local[*]


In [4]:
import shutil

s = spark.range(1_000_000).selectExpr("sum(id) as s").first()["s"]
assert s == 499_999_500_000, f"got {s}"
print("arithmetic OK:", s)

spark.range(1000).write.mode("overwrite").parquet("/kaggle/working/_smoke")
n = spark.read.parquet("/kaggle/working/_smoke").count()
assert n == 1000, f"got {n}"
shutil.rmtree("/kaggle/working/_smoke")
print("parquet I/O OK")
print(" Spark is operational")

arithmetic OK: 499999500000


parquet I/O OK
 Spark is operational


In [ ]:
RAW_ROOT = ("/kaggle/input/datasets/hiranomanabu/"
            "ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset")

STREAMS      = ["ata_read", "ata_write", "mem_exec",
                "mem_readwrite", "mem_read", "mem_write"]
HAS_ENTROPY  = {"ata_write", "mem_readwrite", "mem_write"}
EVENT_DDL    = "sec long, ns double, addr long, size long, entropy double, c6 double"
WINDOW_S     = 1.0

RANSOM_FAMILIES = {"Conti", "LockBit", "Darkside", "Ryuk", "REvil", "WannaCry"}


VAL_HW = {
    "celeron-gen6/ddr4-1600-16g",
    "i3-gen12/ddr4-2133-16g",
}
TEST_HW = {
    "celeron-gen6/ddr4-2133-8g",
    "i3-gen12/ddr4-3200-16g",
    "i3-gen12/ddr4-2666-8g",
}
ZERODAY_FAMILY = "Ryuk"     

print("RAW_ROOT exists:", __import__("os").path.isdir(RAW_ROOT))

RAW_ROOT exists: True


In [ ]:
import os, re, time, pandas as pd

NAME_RE = re.compile(r"^(.+)-(\d{8}_\d{2}-\d{2}-\d{2})$")

def find_execution_dirs(base):
    """Every directory containing at least one recognised stream CSV."""
    hits = []
    for root, _, files in os.walk(base):
        csvs = {f[:-4] for f in files if f.endswith(".csv")}
        if csvs & set(STREAMS):
            hits.append((root, csvs))
    return hits

t0 = time.time()
exec_dirs = find_execution_dirs(RAW_ROOT)
print(f"scanned in {time.time()-t0:.1f}s | {len(exec_dirs)} execution folders")

rows = []
for root, csvs in exec_dirs:
    rel_full = os.path.relpath(root, RAW_ROOT).replace(os.sep, "/")   
    parts = rel_full.split("/")
   
    top, cpu, ram, activity, execution = parts[0], parts[1], parts[2], parts[3], parts[4]

    mt = NAME_RE.match(execution)
    sample = mt.group(1) if mt else execution
    family = re.sub(r"[_-]?\d+$", "", sample)

    sizes = {s: 0 for s in STREAMS}
    for f in os.listdir(root):
        if f.endswith(".csv") and f[:-4] in STREAMS:
            sizes[f[:-4]] = os.path.getsize(os.path.join(root, f))

    rows.append({
        "rel_dir":   rel_full,
        "top":       top, "cpu": cpu, "ram": ram,
        "hw":        f"{cpu}/{ram}",
        "activity":  activity, "execution": execution,
        "sample":    sample, "family": family,
        "has_ata":   int(sizes["ata_read"] > 0 and sizes["ata_write"] > 0),
        "total_bytes": sum(sizes.values()),
    })

m = pd.DataFrame(rows)
m.to_csv("/kaggle/working/manifest_raw.csv", index=False)

print(f"executions: {len(m)} | total size: {m.total_bytes.sum()/1e9:.2f} GB")
print("\ntop:")
print(m.top.value_counts().to_string())
print("\nhw:")
print(m.hw.value_counts().to_string())
print("\nfamilies:")
print(m.family.value_counts().to_string())

scanned in 43.4s | 1970 execution folders
executions: 1970 | total size: 172.08 GB

top:
top
original    1440
extra        360
mix          100
variants      70

hw:
hw
i3-gen12/ddr4-2133-16g        290
i5-gen12/ddr4-2133-16g        120
i3-gen12/ddr5-4800-16g        120
i7-gen12/ddr4-2133-16g        120
celeron-gen6/ddr4-1600-16g    120
celeron-gen6/ddr4-1600-8g     120
celeron-gen6/ddr4-1866-16g    120
celeron-gen6/ddr4-1866-8g     120
celeron-gen6/ddr4-2133-16g    120
celeron-gen6/ddr4-2133-8g     120
i3-gen12/ddr4-2133-8g         120
i3-gen12/ddr4-2666-16g        120
i3-gen12/ddr4-2666-8g         120
i3-gen12/ddr4-3200-16g        120
i3-gen12/ddr4-3200-8g         120

families:
family
Conti                   220
AESCrypt                150
Darkside                150
Firefox                 150
Idle                    150
LockBit                 150
Office                  150
REvil                   150
Ryuk                    150
SDelete                 150
WannaCry               

In [ ]:
import pandas as pd

m = pd.read_csv("/kaggle/working/manifest_raw.csv")

def split_of(row):
   
    if row.family == ZERODAY_FAMILY:            return "zeroday"
    if row.top == "mix":                        return "mix"      
    if row.top == "variants":                   return "variants"  
    if row.top == "extra":                      return "extra"     
   
    if row.hw in TEST_HW:                       return "test"
    if row.hw in VAL_HW:                        return "val"
    return "train"

m["label"]      = m.family.isin(RANSOM_FAMILIES).astype(int)
m["ransomware"] = m.family.where(m.family.isin(RANSOM_FAMILIES), "none")
m["mixed"]      = m.family.str.contains("_").astype(int)
m["split"]      = m.apply(split_of, axis=1)
m["sample_id"]  = m.rel_dir.str.replace("/", "__", regex=False)

print("=== rows per split × label ===")
print(m.groupby(["split", "label"]).size().unstack(fill_value=0))

print("\n=== rows per split × family ===")
print(m.groupby(["split", "family"]).size().unstack(fill_value=0).T)

m.to_csv("/kaggle/working/manifest.csv", index=False)
print("\n saved /kaggle/working/manifest.csv")

=== rows per split × label ===
label       0    1
split             
extra     180  150
mix       100    0
test      180  150
train     420  350
val       120  100
variants    0   70
zeroday     0  150

=== rows per split × family ===
split                 extra  mix  test  train  val  variants  zeroday
family                                                               
AESCrypt                 30    0    30     70   20         0        0
AESCrypt_Conti            0   10     0      0    0         0        0
AESCrypt_REvil            0   10     0      0    0         0        0
Conti                    30    0    30     70   20        70        0
Darkside                 30    0    30     70   20         0        0
Firefox                  30    0    30     70   20         0        0
Firefox_Conti             0   10     0      0    0         0        0
Firefox_Office_REvil      0   10     0      0    0         0        0
Firefox_REvil             0   10     0      0    0         0     

In [ ]:
import os, time, shutil
from functools import reduce
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, DoubleType
from pyspark.sql.window import Window

RAW_ROOT = globals().get("RAW_ROOT", "/kaggle/input/datasets/hiranomanabu/"
                         "ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset")
WORK = globals().get("WORK", "/kaggle/working")
MANIFEST_CSV = f"{WORK}/manifest.csv"
BUILD_SPLITS = ["zeroday", "val", "test", "train"]          
FRACTIONS = [10, 25, 50, 75]

_STREAMS = ["ata_read", "ata_write", "mem_exec", "mem_readwrite", "mem_read", "mem_write"]
_HAS_ENTROPY = {"ata_write", "mem_readwrite", "mem_write"}
_WINDOW_S = 1.0
_META = ["rel_dir", "family", "top", "cpu", "ram", "hw", "label"]
_SCHEMA = StructType([StructField("sec", LongType(), True), StructField("ns", DoubleType(), True),
                      StructField("addr", LongType(), True), StructField("size", LongType(), True),
                      StructField("entropy", DoubleType(), True), StructField("c6", DoubleType(), True)])


def _done(path):
    return os.path.exists(os.path.join(path, "_SUCCESS"))


def _ingest_stream(stream, rel_dirs, out_root):
    out_path = os.path.join(out_root, stream)
    if _done(out_path):
        print(f"  {stream}: already done"); return
    t0 = time.time()
    paths = [os.path.join(RAW_ROOT, r, f"{stream}.csv") for r in rel_dirs]
    missing = [p for p in paths if not os.path.isfile(p)]
    paths = [p for p in paths if os.path.isfile(p)]
    if missing:
        print(f"  WARNING {stream}: {len(missing)} missing file(s), e.g. {missing[0]}")
    raw = (spark.read.schema(_SCHEMA).option("header", "false").csv(paths)
           .withColumn("_source", F.input_file_name()))
    raw = raw.withColumn("rel_dir", F.regexp_extract(F.col("_source"), r"/dataset/(.+)/[^/]+$", 1))
    valid = (F.col("sec").isNotNull() & F.col("ns").isNotNull() & F.col("addr").isNotNull()
             & F.col("size").isNotNull() & (F.col("size") >= 0))
    raw = raw.withColumn("_valid", valid).withColumn(
        "win", F.floor((F.col("sec") + F.col("ns") / 1_000_000_000.0) / _WINDOW_S).cast("long"))
    aggs = [F.sum(F.when(F.col("_valid"), 1).otherwise(0)).alias(f"{stream}_n"),
            F.count("*").alias(f"{stream}_rawcnt"),
            F.sum(F.when(F.col("_valid"), F.col("size")).otherwise(0)).alias(f"{stream}_size_sum"),
            (F.max(F.when(F.col("_valid"), F.col("addr"))) - F.min(F.when(F.col("_valid"), F.col("addr"))))
            .alias(f"{stream}_addr_range")]
    if stream in _HAS_ENTROPY:
        ent = F.col("_valid") & F.col("entropy").isNotNull() & (F.col("entropy") > 0)
        aggs += [F.avg(F.when(ent, F.col("entropy"))).alias(f"{stream}_entropy_mean"),
                 F.max(F.when(ent, F.col("entropy"))).alias(f"{stream}_entropy_max")]
    raw.groupBy("rel_dir", "win").agg(*aggs).write.mode("overwrite").parquet(out_path)
    print(f"  {stream}: done in {time.time() - t0:.0f}s")


def _merge_split(name, man_df):
    out_root, path = f"{WORK}/manifest_{name}", f"{WORK}/merged_{name}"
    if _done(path):
        print(f"  merged_{name}: already done"); return
    dfs = [spark.read.parquet(f"{out_root}/{s}").filter(F.col("win").isNotNull()).drop(f"{s}_rawcnt")
           for s in _STREAMS]
    merged = reduce(lambda a, b: a.join(b, ["rel_dir", "win"], "full_outer"), dfs)
    meta = spark.createDataFrame(man_df[_META].copy())
    win = merged.join(F.broadcast(meta), on="rel_dir", how="inner")
    num = [c for c in win.columns if c not in _META + ["win"]]
    win = win.fillna(0, subset=num)
    win = win.withColumn("step", F.col("win") - F.min("win").over(Window.partitionBy("rel_dir"))).drop("win")
    win.write.mode("overwrite").parquet(path)
    print(f"  merged_{name}: {len(num)} features")


def _processed_split(man, splits, out):
    """Same logic as the sequence-generation notebook (no gap fill) + a `split` column."""
    runs, feat_ref = [], None
    for sp in splits:
        p = f"{WORK}/merged_{sp}"
        if not os.path.exists(p):
            print(f"skip {sp}: not built"); continue
        d = pd.read_parquet(p)
        feat = [c for c in d.columns if c not in _META + ["step"]]
        feat_ref = feat_ref or feat
        assert feat == feat_ref, f"{sp}: feature columns differ"
        assert set(d.rel_dir) <= set(man.loc[man.split == sp, "rel_dir"]), f"{sp}: runs not in manifest split"
        for rid, g in d.sort_values(["rel_dir", "step"]).groupby("rel_dir", sort=False):
            runs.append((rid, g["family"].iloc[0], int(g["label"].iloc[0]), sp, g[feat].to_numpy(np.float64)))
    for f in FRACTIONS:
        rows = []
        for rid, fam, lab, sp, X in runs:
            n = max(1, int(len(X) * f / 100))
            rows.append(dict(rel_dir=rid, family=fam, label=lab, split=sp, sequence=[list(x) for x in X[:n]], seq_len=n))
        o = os.path.join(out, f"{f}pct"); os.makedirs(o, exist_ok=True)
        pd.DataFrame(rows).to_parquet(os.path.join(o, "part-0.parquet"), index=False)
        print(f"  {f}%: {len(rows)} runs -> {o}")
    return runs


_man = pd.read_csv(MANIFEST_CSV)
assert _man[(_man.family == "Ryuk") & _man.split.isin(["train", "val", "test"])].empty, "Ryuk in train/val/test"
t_all = time.time()
for _sp in BUILD_SPLITS:
    _df = _man[_man.split == _sp]
    print(f"\n=== {_sp}: {len(_df)} runs ===")
    for _s in _STREAMS:
        _ingest_stream(_s, list(_df.rel_dir), f"{WORK}/manifest_{_sp}")
    _merge_split(_sp, _df)
print(f"\nextraction finished in {(time.time() - t_all) / 60:.1f} min")

print("\n=== processed_split (with `split` column) ===")
_runs = _processed_split(_man, BUILD_SPLITS, f"{WORK}/processed_split")


_chk = pd.read_parquet(f"{WORK}/processed_split/25pct", columns=["rel_dir", "family", "split", "label"])
print("\nruns per split:", _chk.split.value_counts().to_dict())
print("Ryuk per split:", _chk[_chk.family == "Ryuk"].split.value_counts().to_dict())
for _sp in BUILD_SPLITS:
    _exp = int((_man.split == _sp).sum()); _got = int((_chk.split == _sp).sum())
    print(f"  {_sp}: {_got}/{_exp} runs", "OK" if _got == _exp else "<-- MISSING RUNS (check warnings above)")
shutil.make_archive(f"{WORK}/processed_split", "zip", f"{WORK}/processed_split")
print(f"\nzip: {WORK}/processed_split.zip  ({os.path.getsize(WORK + '/processed_split.zip') / 1e6:.0f} MB)")


=== zeroday: 150 runs ===


  ata_read: done in 46s


  ata_write: done in 38s


  mem_exec: done in 7s


  mem_readwrite: done in 7s


  mem_read: done in 42s


... (18 outputs left out)

  mem_readwrite: done in 26s


  mem_read: done in 580s


  mem_write: done in 652s


  merged_train: 24 features

extraction finished in 84.7 min

=== processed_split (with `split` column) ===
  10%: 1470 runs -> /kaggle/working/processed_split/10pct
  25%: 1470 runs -> /kaggle/working/processed_split/25pct
  50%: 1470 runs -> /kaggle/working/processed_split/50pct
  75%: 1470 runs -> /kaggle/working/processed_split/75pct

runs per split: {'train': 770, 'test': 330, 'val': 220, 'zeroday': 150}
Ryuk per split: {'zeroday': 150}
  zeroday: 150/150 runs OK
  val: 220/220 runs OK
  test: 330/330 runs OK
  train: 770/770 runs OK

zip: /kaggle/working/processed_split.zip  (34 MB)
